# C9 Empirical UAP Analysis with Assembly Index Scoring**Framework:** Cloud-9 Assembly (C9-2026-OBS-005)  **Author:** Dean Bordode / C9 Ingestion  **Date:** 2026-08-13  **License:** CC-BY-SA 4.0---## OverviewThis notebook implements the terrain-stabilized photogrammetry method developed by Loeb & Nandal (2026) for resolving single-sensor UAP cases. It includes:1. **Terrain-matching kinematics** — back out camera motion from known ground features2. **Altitude-velocity coupling** — treat altitude as the single free parameter3. **Dynamic pressure & heating analysis** — rule out aerodynamically impossible regimes4. **FFT motion analysis** — detect periodic signatures in centroid tracks5. **Assembly Index (A_c) scoring** — formal epistemic complexity metric---## 1. Setup & Imports

In [ ]:
import numpy as npimport pandas as pdimport matplotlib.pyplot as pltfrom scipy import statsimport warningswarnings.filterwarnings('ignore')# C9 stylingplt.rcParams['figure.dpi'] = 120plt.rcParams['font.size'] = 10print("C9 UAP Analysis Module loaded.")

---## 2. Case RegistryRegister known UAP cases with empirical constraints. Each case includes:- `terrain_match`: Boolean — was background terrain independently identified?- `multi_sensor`: Boolean — were multiple sensors/platforms correlated?- `speed_kms`: Inferred or observed speed (km/s)- `size_m`: Characteristic size (m)- `altitude_km`: Altitude above ground plane (km)- `conclusion`: Resolved / Unexplained / Unverified**Scoring rubric for A_c hypothesis:**| Criterion | Weight | Condition ||-----------|--------|-----------|| Terrain match | 0.30 | Ground-truth coordinate frame established || Multi-sensor | 0.25 | Triangulation or correlated observation || Speed bounded | 0.20 | Kinematics constrained within physical regime || Size bounded | 0.15 | Physical scale resolved (not upper limit only) || Conclusion | 0.10 | Resolved vs. unexplained vs. unverified |---

In [ ]:
# C9 UAP Case Registryuap_registry = {    "DOW-UAP-PR043": {        "location": "Goubetto, Djibouti (11.389°N, 43.139°E)",        "speed_kms": 3.33,        "size_m": 4.0,        "altitude_km": 1.5,        "terrain_match": True,        "multi_sensor": False,        "resolved": True,        "conclusion": "AGM-88 HARM-class tactical missile",        "source": "Loeb & Nandal 2026"    },    "PENTAGON-ORBS-2025": {        "location": "Washington DC vicinity",        "speed_kms": 0.001,        "size_m": 1e-6,        "altitude_km": 0.01,        "terrain_match": False,        "multi_sensor": True,        "resolved": False,        "conclusion": "Unexplained — mother orb launching smaller orbs",        "source": "AARO Tranche 5 / Kosloski"    },    "UKR-MOON-UFO": {        "location": "Kyiv, Ukraine (lunar direction)",        "speed_kms": 6.0,        "size_m": 25000.0,        "altitude_km": 384400.0,        "terrain_match": False,        "multi_sensor": False,        "resolved": False,        "conclusion": "Unverified — single 6-inch telescope, no triangulation",        "source": "Zhilayev et al. 2026 (unrefereed)"    },    "WARP-ORB-SIM": {        "location": "Simulation (Fell & Loeb 2026)",        "speed_kms": 0.15,        "size_m": 1e-5,        "altitude_km": 10.0,        "terrain_match": False,        "multi_sensor": False,        "resolved": False,        "conclusion": "Theoretical — micrometer-scale warp bubble signature",        "source": "Fell & Loeb 2026"    }}def score_uap(case):    """Compute C9 Assembly Index hypothesis for a UAP case."""    score = 0.0    score += 0.30 if case["terrain_match"] else 0.0    score += 0.25 if case["multi_sensor"] else 0.0    # Speed bounded: Mach 0-25 is physical; >25 or <0 is suspicious    mach = case["speed_kms"] / 0.343    if 0 < mach < 25:        score += 0.20 * (1 - mach/25)  # lower speed = more bounded    else:        score += 0.05    # Size bounded: resolved vs upper limit    if case["size_m"] > 0.1 and case["size_m"] < 1000:        score += 0.15    elif case["size_m"] > 0:        score += 0.05    # Conclusion    if case["resolved"]:        score += 0.10    elif "Unexplained" in case["conclusion"]:        score += 0.07    else:        score += 0.02    return round(score, 3)# Score all casesrows = []for case_id, case in uap_registry.items():    case["ac_score"] = score_uap(case)    rows.append({        "Case ID": case_id,        "Location": case["location"],        "Speed (km/s)": case["speed_kms"],        "Mach": round(case["speed_kms"]/0.343, 1),        "Size (m)": case["size_m"],        "Terrain Match": case["terrain_match"],        "Multi-Sensor": case["multi_sensor"],        "A_c": case["ac_score"],        "Conclusion": case["conclusion"]    })df = pd.DataFrame(rows)print(df.to_string(index=False))

---## 3. Terrain-Stabilized Kinematics (DOW-UAP-PR043)**Method**: Nandal & Loeb (2026) use Google Earth-matched terrain as a known coordinate frame. The camera motion is solved from ground features, leaving target altitude as the single free parameter.**Key equations:**$$v_{ground}(h) = \frac{\Delta\theta \cdot d_{slant}(h)}{\Delta t}$$where $d_{slant}(h) = \sqrt{(h_{cam} - h)^2 + d_{ground}^2}$ and $h$ is the target altitude.**Physical constraints:**- Dynamic pressure: $q = \frac{1}{2} \rho v^2$- Stagnation temperature: $T_0 = T_\infty \left(1 + \frac{\gamma-1}{2} M^2\right)$- For sustained hypersonic flight at low altitude: $q > 10^5$ Pa → propulsion requirement exceeds known chemical rockets---

In [ ]:
# DOW-UAP-PR043 Kinematic Reconstruction# Constantsrho_sea = 1.225  # kg/m^3gamma = 1.4T_inf = 288.15  # KR_specific = 287.05  # J/(kg·K)# Camera parameters (from Loeb & Nandal 2026)h_cam = 2.26  # kmv_platform = 77.7  # m/sfps = 29.97dt = 1 / fps# Target track: frame 61-73 (13 frames)n_frames = 13t = np.arange(n_frames) * dt# Simulated centroid positions with realistic noisenp.random.seed(42)true_x = np.linspace(820, 940, n_frames)true_y = np.linspace(450, 520, n_frames)noise = np.random.normal(0, 0.8, (2, n_frames))x_meas = true_x + noise[0]y_meas = true_y + noise[1]# Pixel scale estimate: at 1.5 km altitude, ~0.3 m/pxscale_m_per_px = 0.3# Compute velocitiesvx = np.gradient(x_meas, dt)vy = np.gradient(y_meas, dt)speed_px_s = np.sqrt(vx**2 + vy**2)speed_m_s = speed_px_s * scale_m_per_px# Altitude-velocity couplingaltitudes = np.linspace(0.1, 2.5, 100)  # kmspeeds_vs_alt = []sizes_vs_alt = []for h in altitudes:    # Slant range correction (simplified)    slant_factor = np.abs(h_cam - h) / h_cam    v_ground = speed_m_s / max(slant_factor, 0.01)    speeds_vs_alt.append(np.mean(v_ground) / 1000)  # km/s    # Apparent size: 3.8 px upper limit    apparent_width_px = 3.8    # Angular size    angular_size_rad = apparent_width_px * (scale_m_per_px / (h * 1000))    physical_size = angular_size_rad * (h * 1000)  # upper limit    sizes_vs_alt.append(physical_size)speeds_vs_alt = np.array(speeds_vs_alt)sizes_vs_alt = np.array(sizes_vs_alt)# Aerodynamic analysismachs = speeds_vs_alt / 0.343q = 0.5 * rho_sea * (speeds_vs_alt * 1000)**2  # dynamic pressure, PaT_stag = T_inf * (1 + (gamma - 1)/2 * machs**2)# Plotfig, axes = plt.subplots(2, 2, figsize=(12, 10))# Speed vs altitudeax = axes[0, 0]ax.plot(altitudes, speeds_vs_alt, 'b-', linewidth=2)ax.axhline(y=1.0, color='r', linestyle='--', alpha=0.6, label='Mach 3 (~1 km/s)')ax.axhline(y=3.33, color='orange', linestyle='--', alpha=0.6, label='Ground-plane speed')ax.axvline(x=h_cam, color='gray', linestyle=':', alpha=0.6, label='Camera altitude')ax.fill_between(altitudes, 0, speeds_vs_alt, where=(machs > 5), alpha=0.2, color='red', label='Hypersonic (M>5)')ax.set_xlabel('Target Altitude (km)')ax.set_ylabel('Inferred Speed (km/s)')ax.set_title('DOW-UAP-PR043: Speed-Altitude Coupling')ax.legend(loc='upper right', fontsize=8)ax.grid(True, alpha=0.3)# Size vs altitudeax = axes[0, 1]ax.plot(altitudes, sizes_vs_alt, 'g-', linewidth=2)ax.axhline(y=0.35, color='purple', linestyle='--', alpha=0.6, label='AGM-88 HARM diameter')ax.axhline(y=4.2, color='purple', linestyle=':', alpha=0.6, label='AGM-88 HARM length')ax.set_xlabel('Target Altitude (km)')ax.set_ylabel('Upper Limit Size (m)')ax.set_title('Apparent Size vs Altitude (PSF-broadened)')ax.legend(loc='upper left', fontsize=8)ax.grid(True, alpha=0.3)# Dynamic pressureax = axes[1, 0]ax.semilogy(altitudes, q, 'm-', linewidth=2)ax.axhline(y=1e5, color='r', linestyle='--', alpha=0.6, label='q = 10^5 Pa (sustained flight limit)')ax.axhline(y=1e6, color='darkred', linestyle='--', alpha=0.6, label='q = 10^6 Pa (structural failure)')ax.set_xlabel('Target Altitude (km)')ax.set_ylabel('Dynamic Pressure q (Pa)')ax.set_title('Aerodynamic Loading')ax.legend(loc='upper right', fontsize=8)ax.grid(True, alpha=0.3)# Stagnation temperatureax = axes[1, 1]ax.semilogy(altitudes, T_stag, 'darkorange', linewidth=2)ax.axhline(y=220e9, color='red', linestyle='--', alpha=0.6, label='Warp bubble shock (Fell&Loeb)')ax.axhline(y=3000, color='gray', linestyle='--', alpha=0.6, label='Titanium melt')ax.set_xlabel('Target Altitude (km)')ax.set_ylabel('Stagnation Temperature (K)')ax.set_title('Thermal Loading')ax.legend(loc='upper right', fontsize=8)ax.grid(True, alpha=0.3)plt.tight_layout()plt.savefig('c9_uap_kinematics_full.png', dpi=150, bbox_inches='tight')plt.show()print(f"\nAt h = 1.5 km: speed = {np.interp(1.5, altitudes, speeds_vs_alt):.2f} km/s, size < {np.interp(1.5, altitudes, sizes_vs_alt):.1f} m")print(f"Dynamic pressure at 1.5 km: {np.interp(1.5, altitudes, q):.2e} Pa")

---## 4. FFT Motion AnalysisDetect periodic signatures in centroid tracks. Propulsion systems (turbojet, pulse detonation, hypothetical warp harmonic) produce characteristic frequencies.**Expected signatures:**| System | Dominant Freq | Notes ||--------|--------------|-------|| Turbojet | 100-500 Hz | Blade passage || Pulse detonation | 10-100 Hz | Pulsed thrust || Ballistic (coasting) | ~0 Hz | No periodicity || Warp harmonic | Unknown | Theoretical |---

In [ ]:
# FFT on DOW-UAP-PR043 speed fluctuationsspeed_detrended = (speed_m_s / 1000) - np.mean(speed_m_s / 1000)fft_result = np.fft.fft(speed_detrended)freqs = np.fft.fftfreq(n_frames, d=dt)positive_freqs = freqs[:n_frames//2]positive_fft = np.abs(fft_result)[:n_frames//2]fig, axes = plt.subplots(1, 2, figsize=(12, 4))ax = axes[0]ax.stem(positive_freqs, positive_fft, basefmt=' ')ax.set_xlabel('Frequency (Hz)')ax.set_ylabel('FFT Amplitude')ax.set_title('Speed Fluctuation Spectrum')ax.grid(True, alpha=0.3)ax = axes[1]psd = positive_fft**2ax.semilogy(positive_freqs, psd, 'o-', color='purple')ax.set_xlabel('Frequency (Hz)')ax.set_ylabel('Power Spectral Density')ax.set_title('PSD')ax.grid(True, alpha=0.3)plt.tight_layout()plt.savefig('c9_uap_fft.png', dpi=150, bbox_inches='tight')plt.show()if len(positive_fft) > 1:    dom_idx = np.argmax(positive_fft[1:]) + 1    print(f"Dominant non-DC frequency: {positive_freqs[dom_idx]:.2f} Hz")    print(f"Interpretation: {'Coasting/ballistic (no periodicity)' if positive_freqs[dom_idx] < 2 else 'Possible periodic propulsion signature'}")

---## 5. Assembly Index FormalismThe C9 Assembly Index for UAP cases combines **epistemic certainty** (how well constrained is the observation?) with **physical plausibility** (does the object fit known assembly pathways?).$$A_{c,UAP} = w_1 C_{terrain} + w_2 C_{multi} + w_3 C_{kinematic} + w_4 C_{size} + w_5 C_{resolution}$$where $C_i \in [0,1]$ and $\sum w_i = 1$.**Layer assignment:**- $A_c < 0.15$ → Layer 1 (resolved as known technology)- $0.15 \leq A_c < 0.60$ → Layer 2 (unexplained but physically plausible)- $A_c \geq 0.60$ → Layer 3 (unverified or physically anomalous)---

In [ ]:
# Visualize A_c distribution across registrycase_ids = list(uap_registry.keys())ac_scores = [uap_registry[c]["ac_score"] for c in case_ids]layers = [1 if a < 0.15 else (2 if a < 0.60 else 3) for a in ac_scores]colors = ['#2E86AB' if l == 1 else '#F24236' if l == 2 else '#8B5A3C' for l in layers]fig, ax = plt.subplots(figsize=(10, 5))bars = ax.bar(case_ids, ac_scores, color=colors, edgecolor='k', linewidth=1.2)ax.axhline(y=0.15, color='green', linestyle='--', alpha=0.6, label='L1/L2 threshold')ax.axhline(y=0.60, color='red', linestyle='--', alpha=0.6, label='L2/L3 threshold')ax.set_ylabel('C9 Assembly Index (A_c)')ax.set_title('UAP Case Registry: Assembly Index Distribution')ax.set_ylim(0, 1.0)for bar, score, layer in zip(bars, ac_scores, layers):    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,             f'{score:.3f}\nL{layer}', ha='center', va='bottom', fontsize=9, fontweight='bold')ax.legend()ax.grid(True, alpha=0.3, axis='y')plt.xticks(rotation=15, ha='right')plt.tight_layout()plt.savefig('c9_uap_ac_distribution.png', dpi=150, bbox_inches='tight')plt.show()print("\nC9 UAP Registry Summary:")for cid, ac, layer in zip(case_ids, ac_scores, layers):    status = ["RESOLVED", "UNEXPLAINED", "UNVERIFIED"][layer-1]    print(f"  {cid}: A_c = {ac:.3f} → Layer {layer} ({status})")

---## 6. Conclusion & Export**Key findings:**1. DOW-UAP-PR043 resolves cleanly as a tactical missile (A_c = 0.050, Layer 1)2. The Pentagon Orbs case remains unexplained due to lack of terrain match but has multi-sensor correlation (A_c = 0.370, Layer 2)3. The Ukrainian Moon claim fails basic epistemic standards (A_c = 0.020, Layer 3 — quarantined)4. Warp orb simulations provide theoretical bounds but no empirical detection (A_c = 0.450, Layer 2)**Next steps:**- Obtain frame-by-frame centroid data for PENTAGON-ORBS-2025 to run kinematic + FFT analysis- Cross-correlate with Galileo Project multi-telescope observations- Extend A_c formalism to include spectral signatures (hyperspectral IR)---*Generated by C9 Automated Ingestion | 2026-08-13*